# FreeFine — Wave 1A Core Resume v2
## Continue ONLY the 142 structural core-metric cells left by the first Wave-1A run
Attach: original GeoBench inputs, original Account-A-v3 output, final Wave-0A resume output, and the saved output of the partial Wave-1A run containing `A_core_results.json`.
Run on T4×2.


## 1. Clone pinned FreeFine


In [1]:

# ===== Clone pinned FreeFine + clock =====
import os,time,subprocess,json,glob,shutil,csv,re,hashlib,random
NB_START=time.time()
COMMIT="4c9fdb971572b32edbeac13464659274c28decbb"
subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && "
    f"cd FreeFine && git checkout -q {COMMIT}",
    shell=True,check=True
)
print("✓ cloned + pinned FreeFine",COMMIT[:14])


✓ cloned + pinned FreeFine 4c9fdb971572b3


## 2. Metric environment


In [2]:

%%bash
set -e
V=/kaggle/temp/metric_env
PY=$V/bin/python
REPO=/kaggle/temp/FreeFine
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do
  cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true
done
echo "✓ metric_env OK"


✓ metric_env OK


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 523ms
 Downloaded torchaudio
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchvision
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded pillow
 Downloaded nvidia-nvjitlink-cu12
 Downloaded networkx
 Downloaded triton
 Downloaded numpy
 Downloaded nvidia-curand-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded sympy
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded torch
Prepared 27 packages in 29.08s
Installed 27 packages in 395ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvid

## 3. Deterministic metric patches


In [3]:

# ===== Deterministic metric patches =====
import pathlib,re,py_compile
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"
s=mp.read_text()
s=s.replace("args.3d","getattr(args,'3d')")
anchor="    args = parser.parse_args()\n"
assert anchor in s
seed_patch=anchor+"""    # Reproducible metric-side RNG (especially KD and model initialization paths)
    import random as _random, numpy as _np
    try:
        import torch as _torch
    except Exception:
        _torch=None
    _metric_seed=int(os.environ.get('FF_METRIC_SEED','42'))
    _random.seed(_metric_seed); _np.random.seed(_metric_seed)
    if _torch is not None:
        _torch.manual_seed(_metric_seed)
        if _torch.cuda.is_available(): _torch.cuda.manual_seed_all(_metric_seed)
"""
s=s.replace(anchor,seed_patch,1)
mp.write_text(s)

for f in [mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace(
        "stabilityai/stable-diffusion-2-1",
        "sd2-community/stable-diffusion-2-1"
    ))

md=mr/"MD"/"mean_distance.py"
s=md.read_text()
needle="all_dist = []"
assert needle in s
s=s.replace(
    needle,
    needle+"\n    import torch as _st, os as _os; "
           "_seed=int(_os.environ.get('FF_MD_SEED','42')); "
           "_st.manual_seed(_seed); _st.cuda.manual_seed_all(_seed)",
    1
)
md.write_text(s)
for f in [mr/"main.py",mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    py_compile.compile(str(f),doraise=True)
print("✓ deterministic metric patches installed")


✓ deterministic metric patches installed


## 4. Exact balanced-200 and expanded groups


In [4]:

# ===== Exact historical balanced-200 + expanded analysis groups =====
import os,glob,json,csv,random,shutil,hashlib
from collections import defaultdict,Counter
GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

def one(pattern,desc):
    xs=glob.glob(pattern,recursive=True)
    if not xs:
        raise FileNotFoundError(
            f"Missing {desc}. Attach the same original GeoBench inputs used by the final generation notebooks. Pattern={pattern}"
        )
    return xs[0]

cc=[c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
    if os.path.isdir(f"{c}/source_img")]
if not cc:
    raise FileNotFoundError("Geo-Bench-2D/source_img not found under /kaggle/input")
CACHE=cc[0]
COARSE=one("/kaggle/input/**/coarse_img/*/*/*.png","coarse_img").split("/coarse_img/")[0]+"/coarse_img"
GENBASE=one("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup","baseline generation directory")
IB=os.path.dirname(os.path.dirname(os.path.dirname(
    one("/kaggle/input/**/inp_img_blended/**/inp_img.png","inp_img_blended")
)))
ANNs=one("/kaggle/input/**/annotation_2d.json","annotation_2d.json")
META=one("/kaggle/input/**/sample_metadata.csv","sample_metadata.csv")

for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}",d)
for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(sc,d)

shutil.copy(ANNs,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META))
      if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]

random.seed(42)
cells=defaultdict(list)
for r in meta:
    cells[(r["edit_type"],r["difficulty"])].append(r)
keys=sorted(cells)
per=200//len(keys)
picked=[]
for k in keys:
    pool=cells[k][:]
    random.shuffle(pool)
    picked += pool[:per]
chosen={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in chosen]
random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]
counts=Counter(r["edit_type"] for r in picked)
assert counts==Counter({"move":67,"resize":67,"rotate":66}), counts
json.dump(picked,open(f"{GEO}/subset_meta.json","w"),indent=2)
sha=hashlib.sha256(
    "\n".join(sorted(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}" for r in picked)).encode()
).hexdigest()
print("balanced-200:",dict(counts))
print("SHA256:",sha)
assert sha=="3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c"

EVALROOT=f"{GEO}/gen_eval"
os.makedirs(EVALROOT,exist_ok=True)
d=f"{EVALROOT}/baseline"
if os.path.lexists(d):
    os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
os.symlink(GENBASE,d)

def krow(r): return (r["da_n"],r["ins_id"],r["case_id"])
def mem(pred): return [krow(r) for r in picked if pred(r)]

groups={"all_200":mem(lambda r:True)}
for diff in ["easy","medium","hard"]:
    groups[f"all_{diff}"]=mem(lambda r,diff=diff:r["difficulty"]==diff)
groups["all_nonhard"]=mem(lambda r:r["difficulty"]!="hard")
for et in ["move","rotate","resize"]:
    groups[f"{et}_all"]=mem(lambda r,et=et:r["edit_type"]==et)
    groups[f"{et}_nonhard"]=mem(lambda r,et=et:r["edit_type"]==et and r["difficulty"]!="hard")
    for diff in ["easy","medium","hard"]:
        groups[f"{et}_{diff}"]=mem(lambda r,et=et,diff=diff:r["edit_type"]==et and r["difficulty"]==diff)

GROUP_ORDER=[
    "move_all","move_easy","move_medium","move_hard","move_nonhard",
    "rotate_all","rotate_easy","rotate_medium","rotate_hard","rotate_nonhard",
    "resize_all","resize_easy","resize_medium","resize_hard","resize_nonhard",
    "all_200","all_easy","all_medium","all_hard","all_nonhard",
]
assert all(g in groups for g in GROUP_ORDER)
print("group sizes:",{g:len(groups[g]) for g in GROUP_ORDER})
print("✓ dataset + expanded groups ready")


balanced-200: {'move': 67, 'resize': 67, 'rotate': 66}
SHA256: 3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c
group sizes: {'move_all': 67, 'move_easy': 23, 'move_medium': 22, 'move_hard': 22, 'move_nonhard': 45, 'rotate_all': 66, 'rotate_easy': 22, 'rotate_medium': 22, 'rotate_hard': 22, 'rotate_nonhard': 44, 'resize_all': 67, 'resize_easy': 22, 'resize_medium': 22, 'resize_hard': 23, 'resize_nonhard': 44, 'all_200': 200, 'all_easy': 67, 'all_medium': 66, 'all_hard': 67, 'all_nonhard': 133}
✓ dataset + expanded groups ready


## 5. Discover generated outputs


In [5]:

# ===== Discover best available Account A raw outputs model-by-model =====
import os,glob,json,shutil
RAW=['R1_ALL', 'HF30_ALL', 'FULL30_ALL', 'MID30_ALL', 'MIDHF30_ALL', 'FIXREF_HF30_ALL', 'FIXREF_MIDHF30_ALL', 'DIRHF30_ALL', 'HF30_PURE_ALL', 'FULL30_PURE_ALL', 'MID30_PURE_ALL', 'MIDHF30_PURE_ALL', 'FIXREF_HF30_PURE_ALL', 'FIXREF_MIDHF30_PURE_ALL', 'DIRHF30_PURE_ALL']
def pc(p):
    return len(glob.glob(p+"/**/*.png",recursive=True)) if os.path.isdir(p) else 0
EVALROOT=f"{GEO}/gen_eval"
roots={}
inventory={}
for tag in RAW:
    cands=glob.glob(f"/kaggle/input/**/finalA_structural/variants/{tag}",recursive=True)
    if not cands:
        inventory[tag]={"pngs":0,"expected":200,"complete":False,"root":None}
        print(f"{tag:30s} MISSING")
        continue
    root=max(cands,key=pc)
    n=pc(root)
    roots[tag]=root
    inventory[tag]={"pngs":n,"expected":200,"complete":n>=200,"root":root}
    print(f"{tag:30s} {n}/200", "✓" if n>=200 else "PARTIAL", root)
    d=f"{EVALROOT}/{tag}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(root,d)
json.dump(inventory,open("/kaggle/working/A_inventory.json","w"),indent=2)
COMPLETE=[t for t in RAW if inventory[t]["complete"]]
PARTIAL=[t for t in RAW if inventory[t]["pngs"] and not inventory[t]["complete"]]
print("COMPLETE",len(COMPLETE),COMPLETE)
print("PARTIAL",PARTIAL)


R1_ALL                         200/200 ✓ /kaggle/input/notebooks/georgiostzamouranis/freefine-final-exhaustive-run-account-a-structu/finalA_structural/variants/R1_ALL
HF30_ALL                       200/200 ✓ /kaggle/input/notebooks/georgiostzamouranis/freefine-final-exhaustive-run-account-a-structu/finalA_structural/variants/HF30_ALL
FULL30_ALL                     200/200 ✓ /kaggle/input/notebooks/georgiostzamouranis/freefine-final-exhaustive-run-account-a-structu/finalA_structural/variants/FULL30_ALL
MID30_ALL                      200/200 ✓ /kaggle/input/notebooks/georgiostzamouranis/01-w0a-resume-only-missing-dirhf-t4x2-v2/finalA_structural/variants/MID30_ALL
MIDHF30_ALL                    200/200 ✓ /kaggle/input/notebooks/georgiostzamouranis/freefine-final-exhaustive-run-account-a-structu/finalA_structural/variants/MIDHF30_ALL
FIXREF_HF30_ALL                200/200 ✓ /kaggle/input/notebooks/georgiostzamouranis/freefine-final-exhaustive-run-account-a-structu/finalA_structural/variant

## 6. Metric helpers


In [6]:

# ===== Metric helpers =====
import os,json,re,subprocess,time,threading,queue,pandas as pd
MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"
EVALROOT=f"{GEO}/gen_eval"
ann=json.load(open(f"{GEO}/annotation_2d.json"))

def manifest(tag,group_name,ids):
    out={}; base=f"{EVALROOT}/{tag}"; used=0
    for d,i,e in ids:
        gp=f"{base}/{d}/{i}/{e}.png"
        if not os.path.exists(gp): continue
        lf=dict(ann[d]["instances"][i][e])
        lf["gen_img_path"]=f"gen_eval/{tag}/{d}/{i}/{e}.png"
        out.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
        used+=1
    p=f"{GEO}/m_{re.sub(r'[^A-Za-z0-9_.-]+','_',tag+'_'+group_name)}.json"
    json.dump(out,open(p,"w"))
    return p,used

def run_metric(manifest_path,task,gpu,md_seed=42,metric_seed=42):
    env=os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES":str(gpu),
        "MPLBACKEND":"Agg",
        "HF_HOME":"/kaggle/temp/hf",
        "TORCH_HOME":"/kaggle/temp/torch",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_MD_SEED":str(md_seed),
        "FF_METRIC_SEED":str(metric_seed),
        "TOKENIZERS_PARALLELISM":"false",
    })
    p=subprocess.run(
        [PY,"main.py","--path",manifest_path,"--use_relative_path","--base_dir",GEO,
         "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task",task,"--level","0"],
        cwd=MET,env=env,capture_output=True,text=True
    )
    txt=p.stdout+p.stderr
    vals={}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","WRAP_E","MD"]:
        h=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",txt)
        if h: vals[k]=round(float(h[-1]),4)
    if p.returncode!=0:
        vals["_rc"]=p.returncode; vals["_tail"]=txt[-1800:]
    return vals

def parallel_jobs(jobs,worker_fn,deadline_hours=11.6):
    q=queue.Queue()
    for x in jobs: q.put(x)
    deadline=NB_START+deadline_hours*3600
    def _w(gpu):
        while time.time()<deadline-120:
            try: job=q.get_nowait()
            except queue.Empty: return
            try: worker_fn(gpu,*job)
            finally: q.task_done()
    ts=[threading.Thread(target=_w,args=(g,),daemon=True) for g in [0,1]]
    [t.start() for t in ts]; [t.join() for t in ts]
    return list(q.queue)
print("✓ metric helpers ready")


✓ metric helpers ready


## 7. Run


In [7]:

# ===== Wave 1A RESUME: load previous partial A_core_results and run ONLY missing groups =====
import copy, glob, json, os, pandas as pd

RESULT_JSON="/kaggle/working/A_core_results.json"
RESULT_CSV="/kaggle/working/A_core_results.csv"
TAGS=["baseline"]+[t for t in COMPLETE if os.path.exists(f"{EVALROOT}/{t}")]
TAGS=list(dict.fromkeys(TAGS))

def score_result_file(p):
    try:
        x=json.load(open(p))
        done=0
        for tag,gd in x.items():
            for g,v in gd.items():
                if all(k in v for k in ["BGC","SUBC","WRAP_E","MD"]) and v.get("complete_group"):
                    done+=1
        return done
    except Exception:
        return -1

cands=glob.glob("/kaggle/input/**/A_core_results.json",recursive=True)
assert cands, (
    "Attach the saved output of the completed/partial Wave-1A notebook as an Input. "
    "It must contain A_core_results.json."
)
best=max(cands,key=score_result_file)
results=json.load(open(best))
print("Loaded previous A results:",best)
print("Completed cells in input:",score_result_file(best))

lock=threading.Lock()

# Baseline gate.
m,u=manifest("baseline","all_200",groups["all_200"])
gate=run_metric(m,"000111100",0,42,42)
assert u==200
assert abs(gate.get("SUBC",-9)-0.9154)<5e-4,gate
assert abs(gate.get("BGC",-9)-0.9657)<8e-4,gate
assert abs(gate.get("WRAP_E",-9)-0.0488)<8e-4,gate
assert abs(gate.get("MD",-9)-7.6023)<0.03,gate
print("✓ baseline core gate",gate)

def is_done(tag,g):
    v=results.get(tag,{}).get(g,{})
    return all(k in v for k in ["BGC","SUBC","WRAP_E","MD"]) and v.get("complete_group")

jobs=[(tag,g) for tag in TAGS for g in GROUP_ORDER if not is_done(tag,g)]
print("A missing jobs before resume:",len(jobs))
print(jobs[:160])

def save():
    with lock:
        json.dump(results,open(RESULT_JSON,"w"),indent=2)
        rows=[]
        for tag,gd in results.items():
            for g,v in gd.items():
                rows.append({"set":tag,"group":g,**v})
        pd.DataFrame(rows).to_csv(RESULT_CSV,index=False)

def work(gpu,tag,g):
    m,u=manifest(tag,g,groups[g])
    if not u: return
    v=run_metric(m,"000111100",gpu,42,42)
    v["n"]=u;v["expected_n"]=len(groups[g]);v["complete_group"]=(u==len(groups[g]))
    with lock:
        results.setdefault(tag,{})[g]=v
    save()
    print(f"[GPU{gpu}] {tag:34s} {g:16s} -> {v}",flush=True)

remaining=parallel_jobs(jobs,work,11.65)
save()

missing=[]
for tag in TAGS:
    for g in GROUP_ORDER:
        if not is_done(tag,g): missing.append((tag,g))
print("MISSING CORE",len(missing),missing[:200])
print("remaining queue",len(remaining))
print("saved",RESULT_JSON,RESULT_CSV)
assert not missing, (
    f"Wave 1A still has {len(missing)} missing cells. Save this output and rerun this same "
    "resume notebook with the newest output attached; it will continue only the remainder."
)
print("✓ WAVE 1A CORE COMPLETE")


Loaded previous A results: /kaggle/input/notebooks/papadonikolas/wave-1a-structural/A_core_results.json
Completed cells in input: 178
✓ baseline core gate {'SUBC': 0.9154, 'BGC': 0.9657, 'WRAP_E': 0.0488, 'MD': 7.6023}
A missing jobs before resume: 142
[('DIRHF30_ALL', 'all_hard'), ('DIRHF30_ALL', 'all_nonhard'), ('HF30_PURE_ALL', 'move_all'), ('HF30_PURE_ALL', 'move_easy'), ('HF30_PURE_ALL', 'move_medium'), ('HF30_PURE_ALL', 'move_hard'), ('HF30_PURE_ALL', 'move_nonhard'), ('HF30_PURE_ALL', 'rotate_all'), ('HF30_PURE_ALL', 'rotate_easy'), ('HF30_PURE_ALL', 'rotate_medium'), ('HF30_PURE_ALL', 'rotate_hard'), ('HF30_PURE_ALL', 'rotate_nonhard'), ('HF30_PURE_ALL', 'resize_all'), ('HF30_PURE_ALL', 'resize_easy'), ('HF30_PURE_ALL', 'resize_medium'), ('HF30_PURE_ALL', 'resize_hard'), ('HF30_PURE_ALL', 'resize_nonhard'), ('HF30_PURE_ALL', 'all_200'), ('HF30_PURE_ALL', 'all_easy'), ('HF30_PURE_ALL', 'all_medium'), ('HF30_PURE_ALL', 'all_hard'), ('HF30_PURE_ALL', 'all_nonhard'), ('FULL30_PURE_